# ============================================
#  Notebook 03 — EDA, Classification & Diagnostic Metrics
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 3: EDA, Classification & Diagnostic Metrics, Inference

**Sections:**
1. EDA — Average total correct vs omitted vs fabricated at observation level, grouped by feature
2. Classification & diagnostic metrics with faceted plots (Radiology vs Pathology)
3. All tables and plots from `main_analysis.py` reproduced in notebook form
4. Inference — one-sided McNemar p-values at observation level, grouped by feature, stratified by domain

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import binomtest
from sklearn.metrics import roc_curve, auc, precision_recall_curve

pd.set_option('display.float_format', lambda x: '%.4f' % x)
sns.set_style('whitegrid')

load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(os.getenv("PROJECT_ROOT",
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca"))
DATA_PRIVATE_DIR = Path(os.getenv("DATA_PRIVATE_DIR", r"C:\Users\jamesr4\loc\data_private"))

DATA_PATH  = DATA_PRIVATE_DIR / "raw" / "merged_llm_summary_validation_datasheet.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src" / "llm_eval_by_human"))

from metrics_utils import (
    compute_confusion_counts, compute_metrics_from_counts,
    bootstrap_ci, element_metric_pvalue,
    binary_predictions_from_annotator, roc_pr_from_binary,
    build_confusion_df_from_counts, mcnemar_exact_from_masks,
    metric_correct_masks,
)

In [ ]:
data = pd.read_excel(DATA_PATH)
string_data = ["NA", "na", "n/a", "N/A", "NA ", " na", " n/a", " N/A"]
data = data.replace(string_data, "N/A")
print(f"Dataset: {data.shape[0]} observations x {data.shape[1]} columns")

In [ ]:
ELEMENTS = {
    "Lesion Size": {"source": "lesion_size_status_source", "human": "lesion_size_status_human", "ai": "lesion_size_status_ai"},
    "Lesion Laterality": {"source": "laterality_status_source", "human": "laterality_status_human", "ai": "laterality_status_ai"},
    "Lesion Location": {"source": "lesion_location_status_source", "human": "lesion_location_status_human", "ai": "lesion_location_status_ai"},
    "Calcifications / Asymmetry": {"source": "calcifications_asymmetry_status_source", "human": "calcifications_asymmetry_status_human", "ai": "calcifications_asymmetry_status_ai"},
    "Additional Enhancement (MRI)": {"source": "additional_enhancement_mri_status_source", "human": "additional_enhancement_mri_status_human", "ai": "additional_enhancement_mri_status_ai"},
    "Extent": {"source": "extent_status_source", "human": "extent_status_human", "ai": "extent_status_ai"},
    "Accurate Clip Placement": {"source": "accurate_clip_placement_status_source", "human": "accurate_clip_placement_status_human", "ai": "accurate_clip_placement_status_ai"},
    "Workup Recommendation": {"source": "workup_recommendation_status_source", "human": "workup_recommendation_status_human", "ai": "workup_recommendation_status_ai"},
    "Lymph Node": {"source": "Lymph node_status_source", "human": "Lymph node_status_human", "ai": "Lymph node_status_ai"},
    "Chronology Preserved": {"source": "chronology_preserved_status_source", "human": "chronology_preserved_status_human", "ai": "chronology_preserved_status_ai"},
    "Biopsy Method": {"source": "biopsy_method_status_source", "human": "biopsy_method_status_human", "ai": "biopsy_method_status_ai"},
    "Invasive Component Size (Pathology)": {"source": "invasive_component_size_pathology_status_source", "human": "invasive_component_size_pathology_status_human", "ai": "invasive_component_size_pathology_status_ai"},
    "Histologic Diagnosis": {"source": "histologic_diagnosis_status_source", "human": "histologic_diagnosis_status_human", "ai": "histologic_diagnosis_status_ai"},
    "Receptor Status": {"source": "receptor_status_source", "human": "receptor_status_human", "ai": "receptor_status_ai"},
}
DOMAINS = {
    "Radiology": ["Lesion Size", "Lesion Laterality", "Lesion Location",
                  "Calcifications / Asymmetry", "Additional Enhancement (MRI)",
                  "Extent", "Accurate Clip Placement", "Workup Recommendation",
                  "Lymph Node", "Chronology Preserved", "Biopsy Method"],
    "Pathology": ["Biopsy Method", "Invasive Component Size (Pathology)",
                  "Histologic Diagnosis", "Receptor Status"],
}
BOOTSTRAP_METRICS = ["accuracy", "sensitivity", "specificity", "ppv", "npv",
                     "precision", "recall", "fabrication_rate", "f1"]

---
## Part 1: EDA — Correct vs Omitted vs Fabricated at Observation Level

### 1.1 Count correct (1), omitted (2), fabricated (3) per observation per feature

In [ ]:
obs_rows = []
for element, cols in ELEMENTS.items():
    s_col, h_col, a_col = cols["source"], cols["human"], cols["ai"]
    for idx, row in data.iterrows():
        source_val = row.get(s_col)
        for annotator, ann_col in [("Human", h_col), ("AI", a_col)]:
            ann_val = row.get(ann_col)
            if pd.isna(source_val) or pd.isna(ann_val):
                status = "missing"
            elif source_val == 1 and ann_val == 1:
                status = "correct"
            elif source_val == 1 and ann_val == 2:
                status = "omitted"
            elif source_val == 1 and ann_val == 3:
                status = "fabricated"
            elif source_val == 0 and ann_val == "N/A":
                status = "correct"  # TN
            else:
                status = "other"
            obs_rows.append({"obs_idx": idx, "Element": element, "Annotator": annotator, "status": status})

df_obs = pd.DataFrame(obs_rows)
print(f"Observation-level status records: {len(df_obs)}")

In [ ]:
# Summary: total correct, omitted, fabricated per element per annotator
eda_summary = (
    df_obs[df_obs["status"].isin(["correct", "omitted", "fabricated"])]
    .groupby(["Element", "Annotator", "status"])
    .size()
    .reset_index(name="count")
)

# Pivot for display
eda_pivot = eda_summary.pivot_table(index=["Element", "Annotator"], columns="status",
                                     values="count", fill_value=0).reset_index()
eda_pivot = eda_pivot[["Element", "Annotator", "correct", "omitted", "fabricated"]]
print("Correct / Omitted / Fabricated per Element per Annotator")
eda_pivot

### 1.2 Faceted Bar Charts — Correct vs Omitted vs Fabricated by Feature (Rad vs Path)

In [ ]:
for domain_name, domain_elements in DOMAINS.items():
    df_dom = eda_pivot[eda_pivot["Element"].isin(domain_elements)].copy()
    n_elements = len(domain_elements)
    fig, axes = plt.subplots(1, 3, figsize=(6 * 3, max(5, n_elements * 0.5)))
    fig.suptitle(f"{domain_name} Features: Correct vs Omitted vs Fabricated",
                 fontsize=16, fontweight="bold", y=1.02)

    for ax_i, metric in enumerate(["correct", "omitted", "fabricated"]):
        ax = axes[ax_i]
        sub = df_dom[["Element", "Annotator", metric]].copy()
        sub_pivot = sub.pivot(index="Element", columns="Annotator", values=metric).reindex(domain_elements)
        x = np.arange(len(sub_pivot))
        width = 0.35
        ax.barh(x - width/2, sub_pivot["Human"], width, label="Human", color="#2c3e50")
        ax.barh(x + width/2, sub_pivot["AI"], width, label="AI", color="#95a5a6")
        ax.set_yticks(x)
        ax.set_yticklabels(sub_pivot.index, fontsize=9)
        ax.set_xlabel("Count")
        ax.set_title(metric.title(), fontsize=14, fontweight="bold")
        ax.legend(loc="lower right")
        ax.grid(axis="x", linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"eda_{domain_name.lower()}_correct_omitted_fabricated.png", dpi=300, bbox_inches="tight")
    plt.show()

### 1.3 Stacked Histogram — Distribution of Correct/Omitted/Fabricated per Observation

In [ ]:
# Per-observation totals
obs_totals = (
    df_obs[df_obs["status"].isin(["correct", "omitted", "fabricated"])]
    .groupby(["obs_idx", "Annotator", "status"])
    .size()
    .reset_index(name="count")
    .pivot_table(index=["obs_idx", "Annotator"], columns="status", values="count", fill_value=0)
    .reset_index()
)

for annotator in ["Human", "AI"]:
    sub = obs_totals[obs_totals["Annotator"] == annotator]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist([sub["correct"], sub["omitted"], sub["fabricated"]],
            bins=range(0, 16), stacked=True, label=["Correct", "Omitted", "Fabricated"],
            color=["#27ae60", "#f39c12", "#e74c3c"], edgecolor="black", alpha=0.85)
    ax.set_xlabel("Count per Observation")
    ax.set_ylabel("Number of Observations")
    ax.set_title(f"{annotator}: Distribution of Correct/Omitted/Fabricated per Observation",
                 fontsize=13, fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"eda_obs_histogram_{annotator.lower()}.png", dpi=300)
    plt.show()

---
## Part 2: Classification & Diagnostic Metrics (from main_analysis.py)

### 2.1 Element-Level Metrics

In [ ]:
### 2.1 Element-Level Metrics

# ── Bootstrap CI via confusion-count resampling ───────────────────────────────
# Uses case-resampling bootstrap: resample rows with replacement, recompute
# confusion counts, then derive the metric. Works correctly for all metrics
# (avoids the per-case indicator approach which is incorrect for specificity,
# NPV, and fabrication_rate given the non-standard TP/FP/FN/TN encoding here).

def _count_bootstrap_ci(
    data: pd.DataFrame,
    source_col: str,
    ann_col: str,
    metric_name: str,
    n_boot: int = 2000,
    alpha: float = 0.05,
    seed: int = 42,
) -> tuple:
    rng = np.random.default_rng(seed)
    n = len(data)
    boot_vals = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        d_b = data.iloc[idx]
        counts = compute_confusion_counts(d_b, source_col, ann_col)
        metrics_b = compute_metrics_from_counts(**counts)
        val = metrics_b.get(metric_name, np.nan)
        if pd.notna(val):
            boot_vals.append(val)
    arr = np.array(boot_vals)
    if len(arr) < 10:
        return (np.nan, np.nan)
    return (
        float(np.percentile(arr, 100 * alpha / 2)),
        float(np.percentile(arr, 100 * (1 - alpha / 2))),
    )

# ── Metrics to bootstrap CI for ──────────────────────────────────────────────
BOOTSTRAP_METRICS = [
    "accuracy", "balanced_accuracy",
    "sensitivity", "specificity",
    "ppv", "npv",
    "fabrication_rate", "f1",
]

# ── McNemar metrics (paired categorical at element level) ─────────────────────
MCNEMAR_METRICS = ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]

element_rows = []
for element_name, cols in ELEMENTS.items():
    source_col, human_col, ai_col = cols["source"], cols["human"], cols["ai"]
    mask = data[[source_col, human_col, ai_col]].notnull().all(axis=1)
    d = data.loc[mask].copy()
    if d.empty:
        continue

    human_counts  = compute_confusion_counts(d, source_col, human_col)
    ai_counts     = compute_confusion_counts(d, source_col, ai_col)
    human_metrics = compute_metrics_from_counts(**human_counts)
    ai_metrics    = compute_metrics_from_counts(**ai_counts)

    # Bootstrap CIs (confusion-count resampling — correct for all metrics)
    ci_h, ci_ai = {}, {}
    for m in BOOTSTRAP_METRICS:
        ci_h[m]  = _count_bootstrap_ci(d, source_col, human_col, m)
        ci_ai[m] = _count_bootstrap_ci(d, source_col, ai_col,   m)

    # McNemar exact p-values (one-sided H1: AI > Human) — correct for paired categorical
    p_vals = {}
    for m in MCNEMAR_METRICS:
        p_vals[m] = element_metric_pvalue(d, source_col, human_col, ai_col, metric_name=m)

    for annotator, counts, metrics, ci in [
        ("Human", human_counts, human_metrics, ci_h),
        ("AI",    ai_counts,    ai_metrics,    ci_ai),
    ]:
        row = {"Element": element_name, "Annotator": annotator}
        row.update({k: v for k, v in counts.items()})
        for m in BOOTSTRAP_METRICS:
            row[m]               = metrics.get(m, np.nan)
            row[f"{m}_ci_low"]   = ci[m][0]
            row[f"{m}_ci_high"]  = ci[m][1]
        for m, pv in p_vals.items():
            row[f"p_mcnemar_{m}"] = pv
        element_rows.append(row)

df_elements = pd.DataFrame(element_rows)
df_elements.to_csv(OUTPUT_DIR / "element_level_metrics.csv", index=False)
print(f"Element-level metrics: {df_elements.shape}")
df_elements.head(4)

### 2.2 Confusion Matrix Heatmaps

In [ ]:
from metrics_utils import plot_confusion_heatmap

# Aggregate confusion across all elements
agg_counts = {"Human": {"TP": 0, "FP": 0, "FN": 0, "TN": 0},
              "AI": {"TP": 0, "FP": 0, "FN": 0, "TN": 0}}
for _, row in df_elements.iterrows():
    ann = row["Annotator"]
    for k in ["TP", "FP", "FN", "TN"]:
        agg_counts[ann][k] += int(row[k])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, ann in zip(axes, ["Human", "AI"]):
    c = agg_counts[ann]
    cm_df = build_confusion_df_from_counts(c["TP"], c["FP"], c["FN"], c["TN"])
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=ax, linewidths=1)
    ax.set_title(f"{ann} — Aggregate Confusion Matrix", fontsize=13, fontweight="bold")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_heatmaps.png", dpi=300, bbox_inches="tight")
plt.show()

### 2.3 Faceted Diagnostic Metrics: Radiology vs Pathology

In [ ]:
diag_metrics = ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]

for domain_name, domain_elements in DOMAINS.items():
    df_dom = df_elements[df_elements["Element"].isin(domain_elements)].copy()
    n_metrics = len(diag_metrics)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"{domain_name}: Diagnostic Metrics — Human vs AI (with 95% CIs)",
                 fontsize=16, fontweight="bold", y=1.02)
    axes = axes.flatten()

    for ax_idx, metric in enumerate(diag_metrics):
        ax = axes[ax_idx]
        human = df_dom[df_dom["Annotator"] == "Human"].set_index("Element").reindex(domain_elements)
        ai = df_dom[df_dom["Annotator"] == "AI"].set_index("Element").reindex(domain_elements)

        x = np.arange(len(domain_elements))
        width = 0.35
        h_vals = human[metric].values.astype(float)
        a_vals = ai[metric].values.astype(float)

        h_err = np.array([
            np.clip(h_vals - human[f"{metric}_ci_low"].values.astype(float), 0, None),
            np.clip(human[f"{metric}_ci_high"].values.astype(float) - h_vals, 0, None),
        ])
        a_err = np.array([
            np.clip(a_vals - ai[f"{metric}_ci_low"].values.astype(float), 0, None),
            np.clip(ai[f"{metric}_ci_high"].values.astype(float) - a_vals, 0, None),
        ])

        ax.bar(x - width/2, h_vals, width, label="Human", color="#2c3e50")
        ax.bar(x + width/2, a_vals, width, label="AI", color="#95a5a6")
        ax.errorbar(x - width/2, h_vals, yerr=h_err, fmt="none", ecolor="black", capsize=3)
        ax.errorbar(x + width/2, a_vals, yerr=a_err, fmt="none", ecolor="black", capsize=3)

        # Add significance stars
        p_col = f"p_mcnemar_{metric}"
        if p_col in human.columns:
            for i, elem in enumerate(domain_elements):
                pval = human.loc[elem, p_col] if elem in human.index else np.nan
                if pd.notna(pval) and pval < 0.05:
                    star = "***" if pval < 0.001 else ("**" if pval < 0.01 else "*")
                    y_max = max(h_vals[i], a_vals[i]) + max(h_err[1, i], a_err[1, i]) + 0.03
                    ax.text(x[i], min(y_max, 1.08), star, ha="center", va="bottom", fontsize=11)

        ax.set_xticks(x)
        ax.set_xticklabels(domain_elements, rotation=45, ha="right", fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_title(metric.replace("_", " ").title(), fontsize=13, fontweight="bold")
        ax.grid(axis="y", linestyle=":", alpha=0.6)
        if ax_idx == 0:
            ax.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"faceted_diagnostic_{domain_name.lower()}.png", dpi=300, bbox_inches="tight")
    plt.show()

### 2.4 Side-by-Side Bar: Total Correct / Omitted / Fabricated per Feature (Faceted by Domain)

In [ ]:
for domain_name, domain_elements in DOMAINS.items():
    df_dom_eda = eda_pivot[eda_pivot["Element"].isin(domain_elements)].copy()
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(domain_elements) * 0.6)),
                              sharey=True)
    fig.suptitle(f"{domain_name}: Total Counts per Feature — Human vs AI",
                 fontsize=15, fontweight="bold", y=1.02)

    for ax_i, annotator in enumerate(["Human", "AI"]):
        ax = axes[ax_i]
        sub = df_dom_eda[df_dom_eda["Annotator"] == annotator].set_index("Element").reindex(domain_elements)
        y = np.arange(len(domain_elements))
        width = 0.25
        ax.barh(y - width, sub["correct"], width, label="Correct", color="#27ae60")
        ax.barh(y, sub["omitted"], width, label="Omitted", color="#f39c12")
        ax.barh(y + width, sub["fabricated"], width, label="Fabricated", color="#e74c3c")
        ax.set_yticks(y)
        ax.set_yticklabels(domain_elements, fontsize=9)
        ax.set_xlabel("Count")
        ax.set_title(annotator, fontsize=13, fontweight="bold")
        ax.legend(loc="lower right")
        ax.grid(axis="x", linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"eda_counts_{domain_name.lower()}_human_vs_ai.png", dpi=300, bbox_inches="tight")
    plt.show()

---
## Part 3: Domain-Level Aggregated Metrics

In [ ]:
domain_rows = []
for domain_name, domain_elements in DOMAINS.items():
    for annotator in ["Human", "AI"]:
        sub = df_elements[(df_elements["Annotator"] == annotator) &
                          (df_elements["Element"].isin(domain_elements))]
        agg = {"Domain": domain_name, "Annotator": annotator}
        agg["TP"] = int(sub["TP"].sum())
        agg["FP"] = int(sub["FP"].sum())
        agg["FN"] = int(sub["FN"].sum())
        agg["TN"] = int(sub["TN"].sum())
        metrics = compute_metrics_from_counts(agg["TP"], agg["FP"], agg["FN"], agg["TN"])
        agg.update(metrics)
        domain_rows.append(agg)

df_domain = pd.DataFrame(domain_rows)
df_domain.to_csv(OUTPUT_DIR / "domain_level_aggregated_metrics.csv", index=False)
print("Domain-level aggregated metrics:")
df_domain

In [ ]:
# Domain-level bar chart
metric_labels = ["Accuracy", "Sensitivity", "Specificity", "PPV", "NPV", "Fab. Rate"]
metric_keys = ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax_i, (domain_name, _) in enumerate(DOMAINS.items()):
    ax = axes[ax_i]
    dom = df_domain[df_domain["Domain"] == domain_name]
    h = dom[dom["Annotator"] == "Human"][metric_keys].values.flatten()
    a = dom[dom["Annotator"] == "AI"][metric_keys].values.flatten()
    x = np.arange(len(metric_keys))
    width = 0.35
    ax.bar(x - width/2, h, width, label="Human", color="#2c3e50")
    ax.bar(x + width/2, a, width, label="AI", color="#95a5a6")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, rotation=35, ha="right")
    ax.set_ylim(0, 1.1)
    ax.set_title(domain_name, fontsize=14, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.suptitle("Domain-Level Metrics: Human vs AI", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "domain_aggregated_diagnostic_metrics_human_vs_ai.png", dpi=300, bbox_inches="tight")
plt.show()

---
## Part 4: Inference — McNemar P-Values

### 4.1 Element-Level P-Values Table

In [ ]:
# Extract p-values from element-level results
p_cols = [c for c in df_elements.columns if c.startswith("p_mcnemar_")]
df_pvals = df_elements[df_elements["Annotator"] == "Human"][["Element"] + p_cols].copy()
df_pvals.columns = [c.replace("p_mcnemar_", "") if c != "Element" else c for c in df_pvals.columns]

def sig_star(p):
    if pd.isna(p): return ""
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return ""

for col in df_pvals.columns:
    if col != "Element":
        df_pvals[f"{col}_star"] = df_pvals[col].apply(sig_star)

print("Element-Level One-Sided McNemar P-Values (H1: AI > Human)")
print("* p<0.05  ** p<0.01  *** p<0.001")
df_pvals

### 4.2 P-Values Stratified by Domain

In [ ]:
for domain_name, domain_elements in DOMAINS.items():
    print(f"\n{'=' * 60}")
    print(f"Domain: {domain_name}")
    print(f"{'=' * 60}")
    dom_pvals = df_pvals[df_pvals["Element"].isin(domain_elements)]
    for _, row in dom_pvals.iterrows():
        print(f"  {row['Element']:40s}", end="")
        for metric in ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]:
            if metric in row:
                pv = row[metric]
                star = row.get(f"{metric}_star", "")
                print(f"  {metric[:6]}={pv:.4f}{star}", end="")
        print()

### 4.3 Fabrication Rate Focus: Element-Level

In [ ]:
fab_cols = ["Element", "Annotator", "FP", "TN", "fabrication_rate",
            "fabrication_rate_ci_low", "fabrication_rate_ci_high", "p_mcnemar_fabrication_rate"]
fab_available = [c for c in fab_cols if c in df_elements.columns]
df_fab = df_elements[fab_available].copy()
print("Fabrication Rate by Element & Annotator (with McNemar p-value H1: AI > Human)")
df_fab

In [ ]:
# Save all tables
df_pvals.to_csv(OUTPUT_DIR / "element_pvalues_one_sided.csv", index=False)
df_fab.to_csv(OUTPUT_DIR / "fabrication_rate_element_level.csv", index=False)
df_domain.to_csv(OUTPUT_DIR / "domain_level_aggregated_metrics.csv", index=False)

print("All tables and plots saved to:", OUTPUT_DIR)

---
## Part 5: Overall Mean Metric Comparison — Paired Statistical Tests

**Statistical rationale:**

The 14 clinical elements provide one Human and one AI metric value each (e.g., accuracy for "Lesion Size → Human" vs "Lesion Size → AI"). This **is** correctly treated as a **paired design**:

- The same ~200 patient cases underlie both annotators' scores for every element
- Element-level difficulty (e.g., receptor status is inherently harder to extract than laterality) is a matched confounder — pairing by element removes it
- Even though each observation was rated once (not longitudinally), the pairing is cross-sectional and valid: each element is the unit, and both annotator conditions are applied to the exact same data source

**Test selection:**
- Normality assessed on differences (Shapiro-Wilk, n=14 pairs — small sample, so Wilcoxon is often preferred regardless)
- If differences are approximately normal: **paired t-test** (one-sided H1: AI > Human or AI < Human depending on metric direction)
- If skewed: **Wilcoxon signed-rank** (one-sided)
- Fabrication rate: lower is better → H1: AI < Human (one-sided, direction reversed)

**Contrast with McNemar (Part 4):**
McNemar operates at the *case level within each element* (binary correct/incorrect per observation, paired by patient). The paired t-test/Wilcoxon here operates at the *element level* (metric point estimates, paired by element). These answer different questions: McNemar tests whether AI and human differ for a specific element; the paired test here tests whether AI and human differ on average across all elements.

In [ ]:
from scipy.stats import shapiro, ttest_rel, wilcoxon, mannwhitneyu

# Metrics where HIGHER = better (H1: AI > Human, one-sided right)
HIGHER_BETTER = ["accuracy", "balanced_accuracy", "sensitivity", "specificity",
                 "ppv", "npv", "f1"]
# Metrics where LOWER = better (H1: AI < Human, one-sided left)
LOWER_BETTER  = ["fabrication_rate"]

ALL_PAIRED_METRICS = HIGHER_BETTER + LOWER_BETTER

overall_test_rows = []
for metric in ALL_PAIRED_METRICS:
    if metric not in df_elements.columns:
        continue

    human_s = (df_elements[df_elements["Annotator"] == "Human"]
               .set_index("Element")[metric])
    ai_s    = (df_elements[df_elements["Annotator"] == "AI"]
               .set_index("Element")[metric])
    common  = human_s.index.intersection(ai_s.index)
    h = human_s.loc[common].values.astype(float)
    a = ai_s.loc[common].values.astype(float)

    valid = ~(np.isnan(h) | np.isnan(a))
    h, a  = h[valid], a[valid]
    diffs = a - h          # positive = AI > Human
    n_pairs = len(diffs)

    # Normality of differences
    if n_pairs >= 3:
        sw_stat, sw_p = shapiro(diffs)
    else:
        sw_stat, sw_p = np.nan, np.nan
    is_normal = bool(sw_p > 0.05) if pd.notna(sw_p) else False

    # Direction of H1
    alternative = "greater" if metric in HIGHER_BETTER else "less"

    # Select test
    if is_normal and n_pairs >= 3:
        # Paired t-test — convert two-sided p to one-sided
        t_stat, p_two = ttest_rel(a, h)
        if alternative == "greater":
            p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2
        else:
            p_one = p_two / 2 if t_stat < 0 else 1 - p_two / 2
        test_name = "Paired t-test"
        stat = t_stat
    elif n_pairs >= 3:
        try:
            stat, p_one = wilcoxon(a, h, alternative=alternative)
            test_name = "Wilcoxon signed-rank"
        except ValueError:
            stat, p_one = np.nan, np.nan
            test_name = "Wilcoxon (insufficient variation)"
    else:
        stat, p_one = np.nan, np.nan
        test_name = "Insufficient pairs"

    def _star(p):
        if pd.isna(p): return "—"
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return "ns"

    overall_test_rows.append({
        "Metric":                  metric,
        "Direction":               "↑ higher better" if metric in HIGHER_BETTER else "↓ lower better",
        "N_elements":              n_pairs,
        "Human_mean":              float(np.nanmean(h)),
        "AI_mean":                 float(np.nanmean(a)),
        "Mean_diff_AI_minus_Human": float(np.nanmean(diffs)),
        "Shapiro_p_on_diffs":      round(float(sw_p), 4) if pd.notna(sw_p) else np.nan,
        "Normal_diffs":            is_normal,
        "Test":                    test_name,
        "Stat":                    round(float(stat), 4) if pd.notna(stat) else np.nan,
        "P_one_sided":             round(float(p_one), 4) if pd.notna(p_one) else np.nan,
        "Sig":                     _star(p_one),
    })

df_overall_tests = pd.DataFrame(overall_test_rows)
df_overall_tests.to_csv(OUTPUT_DIR / "overall_mean_metric_paired_tests.csv", index=False)

print("Overall Mean Metric Comparison — Paired Tests (one-sided)")
print("H1: AI > Human for accuracy/sensitivity/… ; AI < Human for fabrication_rate")
print("* p<0.05  ** p<0.01  *** p<0.001  ns=not significant\n")
df_overall_tests.set_index("Metric")

---
## Part 6: Balanced vs Unbalanced Accuracy

**When each matters:**

| Scenario | Prefer |
|---|---|
| Source=1 (present) >> Source=0 (absent) — most elements | **Balanced accuracy** — standard accuracy inflated by majority class |
| Source=0 is reasonably common | Both similar; report both |
| TN < 5 (very few negative cases) | Neither reliable for specificity; rely on sensitivity + PPV |

**In this dataset:** Most clinical elements are present in the source documents for most cases (e.g., biopsy method, laterality). Source=0 cases are rare → standard accuracy is dominated by sensitivity, not specificity. Balanced accuracy = (Se + Sp)/2 equally weights both classes and is the more defensible primary metric.

**Note on `balanced_accuracy` in `compute_metrics_from_counts`:** already computed in `metrics_utils.py` and included in `df_elements` from cell 14.

In [ ]:
# Compare accuracy vs balanced_accuracy per element; flag imbalance
acc_cols = ["Element", "Annotator", "TP", "FP", "FN", "TN", "accuracy", "balanced_accuracy"]
acc_compare = df_elements[[c for c in acc_cols if c in df_elements.columns]].copy()

acc_compare["N_positive"] = acc_compare["TP"] + acc_compare["FN"]   # source=1 cases
acc_compare["N_negative"] = acc_compare["TN"] + acc_compare["FP"]   # source=0 cases (TN) + fabricated (FP)
acc_compare["pos_neg_ratio"] = (
    acc_compare["N_positive"] / acc_compare["N_negative"].replace(0, np.nan)
)
acc_compare["imbalance_flag"] = acc_compare["pos_neg_ratio"].apply(
    lambda r: "Pos >> Neg (>3:1)" if r > 3
    else ("Neg >> Pos (>3:1)" if r < 1/3 else "Balanced (≤3:1)")
)
acc_compare["acc_vs_balanced"] = acc_compare["accuracy"] - acc_compare["balanced_accuracy"]
acc_compare["recommendation"] = acc_compare.apply(
    lambda r: "Use balanced_accuracy"
    if r["imbalance_flag"] != "Balanced (≤3:1)" and r["N_negative"] >= 5
    else ("Unreliable Sp (TN<5) — use Se+PPV" if r["N_negative"] < 5 else "Both similar"),
    axis=1
)

display_cols = ["Element", "Annotator", "N_positive", "N_negative",
                "imbalance_flag", "accuracy", "balanced_accuracy",
                "acc_vs_balanced", "recommendation"]
acc_display = acc_compare[display_cols].sort_values(["Element", "Annotator"])
acc_display.to_csv(OUTPUT_DIR / "accuracy_vs_balanced_accuracy.csv", index=False)

print("Standard vs Balanced Accuracy — class imbalance assessment")
print("acc_vs_balanced > 0: standard accuracy inflated above balanced accuracy\n")
acc_display

In [ ]:
# Scatter: standard accuracy vs balanced accuracy per element (AI annotator)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax_i, annotator in enumerate(["Human", "AI"]):
    sub = acc_compare[acc_compare["Annotator"] == annotator].copy()
    colors = sub["imbalance_flag"].map({
        "Pos >> Neg (>3:1)": "#e74c3c",
        "Neg >> Pos (>3:1)": "#3498db",
        "Balanced (≤3:1)":   "#2ecc71",
    })
    ax = axes[ax_i]
    ax.scatter(sub["accuracy"], sub["balanced_accuracy"],
               c=colors, s=100, edgecolors="black", linewidths=0.6, zorder=3)
    # Reference diagonal
    lims = [0, 1.05]
    ax.plot(lims, lims, "k--", linewidth=0.8, alpha=0.4, label="y = x (equal)")
    for _, row in sub.iterrows():
        ax.annotate(row["Element"], (row["accuracy"], row["balanced_accuracy"]),
                    fontsize=6.5, ha="left", va="bottom",
                    xytext=(3, 3), textcoords="offset points")
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
    ax.set_xlabel("Standard Accuracy")
    ax.set_ylabel("Balanced Accuracy = (Se + Sp) / 2")
    ax.set_title(f"{annotator}: Accuracy vs Balanced Accuracy", fontsize=13, fontweight="bold")
    ax.grid(linestyle=":", alpha=0.5)

    # Custom legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c',
               markersize=8, label='Pos >> Neg (>3:1)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71',
               markersize=8, label='Balanced (≤3:1)'),
        Line2D([0], [0], linestyle='--', color='black', alpha=0.4, label='y = x'),
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc="lower right")

plt.suptitle("Points above diagonal: balanced accuracy > standard accuracy (inflated by class imbalance)",
             fontsize=10, style="italic", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_vs_balanced_accuracy_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

---
## Part 7: Case-Level Metrics by Complexity

Per-case accuracy = proportion of elements correctly extracted out of total applicable elements.
Cases are grouped by a complexity indicator if present in the dataset (`is_complex`, `complexity`,
`case_type`), otherwise a proxy is computed: cases where ≥ 1 element has source=1 across all 14
elements with total source=1 count ≥ median are labeled "High Complexity".

**Statistical test:** Mann-Whitney U (non-parametric, unpaired groups) comparing per-case accuracy
between complex vs non-complex cases for Human and AI separately. Also Wilcoxon signed-rank on
per-case (AI accuracy − Human accuracy) to test whether AI-Human gap differs by complexity group.

In [ ]:
# ── Detect complexity column or build proxy ───────────────────────────────────
COMPLEXITY_COL_CANDIDATES = ["is_complex", "complexity", "case_type", "case_complexity",
                              "complex", "clinical_complexity"]
complexity_col = next((c for c in COMPLEXITY_COL_CANDIDATES if c in data.columns), None)

if complexity_col:
    print(f"Found complexity column: '{complexity_col}'")
    print(data[complexity_col].value_counts())
    complex_flag = data[complexity_col].astype(str).str.lower().isin(
        ["1", "true", "yes", "complex", "high"]
    )
else:
    print("No complexity column found — computing proxy: number of source=1 elements per case.")
    source_cols = [cols["source"] for cols in ELEMENTS.values()]
    n_present = data[source_cols].apply(pd.to_numeric, errors="coerce").eq(1).sum(axis=1)
    median_present = n_present.median()
    complex_flag = n_present >= median_present
    data["_proxy_n_elements_present"] = n_present
    data["_proxy_complexity"]         = complex_flag.map({True: "High Complexity", False: "Low Complexity"})
    print(f"Median elements present per case: {median_present}")
    print(data["_proxy_complexity"].value_counts())
    complexity_col = "_proxy_complexity"

# ── Per-case element-level accuracy ──────────────────────────────────────────
case_rows = []
for idx, row in data.iterrows():
    for annotator in ["Human", "AI"]:
        correct_count = 0
        total_count   = 0
        for element_name, cols in ELEMENTS.items():
            s_col = cols["source"]
            a_col = cols["human"] if annotator == "Human" else cols["ai"]
            src_val = row.get(s_col)
            ann_val = row.get(a_col)
            if pd.isna(src_val) or pd.isna(ann_val):
                continue
            total_count += 1
            src_num = pd.to_numeric(src_val, errors="coerce")
            ann_num = pd.to_numeric(ann_val, errors="coerce")
            # Correct: TP (src=1 & ann=1) or TN (src=0 & ann="N/A")
            if (src_num == 1 and ann_num == 1) or (src_num == 0 and str(ann_val) == "N/A"):
                correct_count += 1
        if total_count > 0:
            case_rows.append({
                "obs_idx":    idx,
                "Annotator":  annotator,
                "n_correct":  correct_count,
                "n_total":    total_count,
                "case_accuracy": correct_count / total_count,
                "complexity": data.loc[idx, complexity_col],
            })

df_case = pd.DataFrame(case_rows)
print(f"\nPer-case records: {len(df_case)}")
df_case.head(6)

In [ ]:
from scipy.stats import mannwhitneyu

# ── Visualize per-case accuracy by complexity ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
complexity_levels = sorted(df_case["complexity"].dropna().unique())
colors_complex = ["#2ecc71", "#e74c3c"] if len(complexity_levels) == 2 else None

for ax_i, annotator in enumerate(["Human", "AI"]):
    ax = axes[ax_i]
    sub = df_case[df_case["Annotator"] == annotator]
    groups = [sub[sub["complexity"] == g]["case_accuracy"].dropna().values
              for g in complexity_levels]
    ax.boxplot(groups, labels=complexity_levels, patch_artist=True,
               boxprops=dict(facecolor="#95a5a6" if annotator == "AI" else "#2c3e50",
                             alpha=0.6),
               medianprops=dict(color="black", linewidth=2))
    # Overlay jittered points
    for j, (grp, vals) in enumerate(zip(complexity_levels, groups)):
        x_jitter = np.random.default_rng(j).uniform(-0.15, 0.15, size=len(vals))
        ax.scatter(j + 1 + x_jitter, vals, s=18, alpha=0.5,
                   color="#2c3e50" if annotator == "Human" else "#7f8c8d", zorder=3)
    ax.set_xlabel("Complexity Group")
    ax.set_ylabel("Per-Case Accuracy (proportion elements correct)")
    ax.set_title(f"{annotator}: Case Accuracy by Complexity", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.1)
    ax.grid(axis="y", linestyle=":", alpha=0.5)

    # Mann-Whitney U between complexity groups
    if len(groups) == 2 and all(len(g) > 0 for g in groups):
        stat_mw, p_mw = mannwhitneyu(groups[0], groups[1], alternative="two-sided")
        ax.text(0.98, 0.02, f"Mann-Whitney U p={p_mw:.4f}",
                transform=ax.transAxes, ha="right", va="bottom",
                fontsize=9, color="navy")

plt.suptitle("Per-Case Element-Level Accuracy: Complex vs Non-Complex Cases",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "case_accuracy_by_complexity.png", dpi=300, bbox_inches="tight")
plt.show()

# ── AI-Human accuracy gap by complexity ──────────────────────────────────────
df_case_wide = df_case.pivot_table(
    index="obs_idx", columns="Annotator", values="case_accuracy"
).reset_index()
df_case_wide.columns.name = None
df_case_wide = df_case_wide.merge(
    df_case[["obs_idx", "complexity"]].drop_duplicates(),
    on="obs_idx", how="left"
)
df_case_wide["ai_minus_human"] = df_case_wide["AI"] - df_case_wide["Human"]

print("\nAI − Human accuracy gap by complexity group:")
print(df_case_wide.groupby("complexity")["ai_minus_human"].describe().round(4))

# Wilcoxon on gap within each group (H0: gap = 0)
print("\nWilcoxon signed-rank: AI − Human gap ≠ 0 within each complexity group")
for grp in complexity_levels:
    gaps = df_case_wide[df_case_wide["complexity"] == grp]["ai_minus_human"].dropna()
    if len(gaps) >= 3:
        try:
            st, p = wilcoxon(gaps, alternative="two-sided")
            print(f"  {grp}: n={len(gaps)}, stat={st:.2f}, p={p:.4f}"
                  f" {'*' if p<0.05 else ''}")
        except ValueError as e:
            print(f"  {grp}: {e}")

df_case_wide.to_csv(OUTPUT_DIR / "case_accuracy_by_complexity.csv", index=False)